In [9]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [10]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [11]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 273 nodes, deleted 253 relationships, completed after 245 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 3587 ms.


### Node Rules 

In [12]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env.json")
# env = Environment("../dtgraph/type_checking/env_common_movies.json")

# env = Environment(
#     "../dtgraph/type_checking/env.json",
#     functions_path="../dtgraph/type_checking/functions.json"
# )

#################################################
old_node_rule = Rule("""
MATCH (p:Person)
GENERATE
(x = (p):Actor {
    name = p.name,
    born = p.born,
    score = p.born * (p.born + 2),
    experienced = p.born < 1950
})
""", env=env, type_strict=True)

old_edge_rule = Rule(
    """
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name,
    years = [y.released],
    movies = [y.title] 
})
""",
    env=env,
    type_strict=True,
)

Rule_HAS_A = Rule(
"""
MATCH (m:Movie)<-[:ACTED_IN]-(p:Person)
GENERATE
((m):Movie {
    movie = m.title
})-[():HAS_A {
    movies = [p.name]
}]->(():Actor{
name = p.name })
""",
env=env,
type_strict=True,
)


Rule_PAIR = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[(x,y):ACTED_IN_PAIR {
    movies = [y.title],
    years = [y.released]
}]->((y):Movie {
    movie = y.title
})
""",
env=env,
type_strict=True,
)
#################################################

Rule_TEST2 = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
(u = (x):Person {
    name = x.name
})-[():IS_AN {
    years = [y.released],
    movies = [y.title]
}]->(z = ():Actor)
""",env=env,type_strict=True,
)

#################################################

Rule_one = Rule(
"""
MATCH (p:Person)
GENERATE
((_):Actor {
    name = p.name
})
""",
env=env,
type_strict=True,
)

Rule_two = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[():WORKED_ON {
    movie = y.title
}]->((y):Movie {
    movie = y.title
})
""",
env=env,
type_strict=True,
)

Rule_Three = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[():IS_AN {
    movies = [y.title]
}]->((_):ActorGroup)
""",
env=env,
type_strict=True,
)


Rule_four = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((_):TempActor {
    name = x.name
})-[():TEMP_REL {
    movie = y.title
}]->((_):TempMovie {
    movie = y.title
})
""",
env=env,
type_strict=True,
)
#############################################


####
test_one_A = Rule(
"""
MATCH (p:Person)
GENERATE
(("const1"):TestA {
    value = "A"
})
""",
env=env,
type_strict=False,
)

test_one_B = Rule(
"""
MATCH (p:Person)
GENERATE
(("const1"):TestB {
    value = "B"
})
""",
env=env,
type_strict=False,
)
####

####### Auxiliary Functions #######

Rule_APOC = Rule(
"""
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)
GENERATE
((x):Actor {
    name = x.name
})-[():ACTED_IN]->((y):Movie {
    movie = apoc.text.upperCamelCase(y.title),
    releaseDate = toString(y.released)
})
""",
env=env,
type_strict=True,
)



### Execute Rules

In [13]:
my_transform = Transformation([Rule_APOC])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 199 ms.

--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (x:Person)-[:ACTED_IN]->(y:Movie)', 'constructors': [{'src': {'ids': ['x'], 'labels': ['Actor'], 'properties': [{'key': 'name', 'value': 'x.name\n'}]}, 'edge': {'ids': [], 'labels': ['ACTED_IN']}, 'tgt': {'ids': ['y'], 'labels': ['Movie'], 'properties': [{'key': 'movie', 'value': 'apoc.text.upperCamelCase(y.title)'}, {'key': 'releaseDate', 'value': 'toString(y.released)\n'}]}}]}
AST:
PropertyAccess
    ├── var: x
    └── prop: name
AST:
FunctionCall: apoc.text.upperCamelCase
    └── PropertyAccess
        ├── var: y
        └── prop: title
AST:
FunctionCall: toString
    └── PropertyAccess
        ├── var: y
        └── prop: released
Rule: Added 280 labels, created 140 nodes, set 828 properties, created 172 relationships, completed after 2078 ms.


2078

### Abort Transformation

In [8]:
my_transform.abort()

KeyboardInterrupt: 